# Trajectory Container Tools - Direct Instanciation Usage Example

This notebook demonstrates the essential usage of Trajectory Container Tools (TCT) dataclasses for creating and working with trajectory data.


## 1. Basic Trajectory Creation

### Simple Trajectory Container Structure

The most straightforward way to create a trajectory is by extending `BaseTrajectoryDataclass`:

In [1]:
from dataclasses import dataclass
from trajectory_container_tools.trj_dataclasses.base_trajectory_dataclass import BaseTrajectoryDataclass
import numpy as np

# Generate sample trajectory data
timesteps = 50
time_array = np.linspace(0, 5, timesteps)
x_positions = np.sin(time_array)
y_positions = np.cos(time_array)
timestamps = np.arange(timesteps) * 0.1  # 10 Hz sampling

@dataclass()
class Simple2DCoordinateTrajectory(BaseTrajectoryDataclass):
    x: np.ndarray
    y: np.ndarray
    timestamps: np.ndarray

# Create trajectory container
trajectory = Simple2DCoordinateTrajectory(
    feature_name="2D coordinate",
    x=x_positions,
    y=y_positions,
    timestamps=timestamps
)

print((
        f"Created trajectory with {trajectory.trajectory_len} timesteps\n"
        f"Available dimensions: {trajectory.get_dimension_names()}\n"
), trajectory)


Created trajectory with 50 timesteps
Available dimensions: ('x', 'y', 'timestamps')
 
          Simple2DCoordinateTrajectory(
             feature_name: 2D coordinate
             trajectory_len: 50
             transposed: False
             dimensions:
                timesteps_indices: (ndarray) shape (50,) range 0 ⟶ 49
                x: (ndarray) shape (50,) range -0.9998286683840896 ⟶ 0.9991927284190055
                y: (ndarray) shape (50,) range -0.9997651572585272 ⟶ 1.0
                timestamps: (ndarray) shape (50,) range 0.0 ⟶ 4.9
          )


### Nested Trajectory Container Structure

For more complex data organization, use `NestedBaseTrajectoryDataclass` for custom implementation or use dataclasses from `primitive_dataclass` module:

In [2]:
from trajectory_container_tools.trj_dataclasses.base_trajectory_dataclass import NestedBaseTrajectoryDataclass
from trajectory_container_tools.trj_dataclasses.primitive_dataclass import Vector2D


@dataclass()
class CustomPoseContainer(NestedBaseTrajectoryDataclass):
    x: np.ndarray
    y: np.ndarray


@dataclass()
class ComplexTrajectory(BaseTrajectoryDataclass):
    timestamps: np.ndarray
    position: CustomPoseContainer
    velocity: Vector2D


sin_cos_trajectory_object = ComplexTrajectory(feature_name="mock",
                                              timestamps=timestamps,
                                              position=CustomPoseContainer(x_positions, y_positions),
                                              velocity=Vector2D(np.ones_like(x_positions), np.ones_like(y_positions))
                                              )

print(sin_cos_trajectory_object)


          ComplexTrajectory(
             feature_name: mock
             trajectory_len: 50
             transposed: False
             dimensions:
                timesteps_indices: (ndarray) shape (50,) range 0 ⟶ 49
                timestamps: (ndarray) shape (50,) range 0.0 ⟶ 4.9
                position:          
                    CustomPoseContainer(
                          x: (ndarray) shape (50,) range -0.9998286683840896 ⟶ 0.9991927284190055
                          y: (ndarray) shape (50,) range -0.9997651572585272 ⟶ 1.0
                    )
                velocity:          
                    Vector2D(
                          x: (ndarray) shape (50,) range 1.0 ⟶ 1.0
                          y: (ndarray) shape (50,) range 1.0 ⟶ 1.0
                    )
          )


## 2. Factory-Based Creation

TCT provides factory functions for dynamic trajectory dataclass creation:

In [3]:

from trajectory_container_tools.utils.factory import (
    TrjDataClassFeatureSpecification,
    trajectory_dataclass_factory
    )

mock_data = np.random.randn(100, 4)  # 100 timesteps, 4 dimensions

# Define the specification
spec = TrjDataClassFeatureSpecification(
        new_feature_dataclass_type='DynamicTrajectory',
        dimension_names=('x', 'y', 'velocity', 'acceleration')
        )

# Create the dataclass type
DynamicTrajectory = trajectory_dataclass_factory(specification=spec)

# Use the dynamically created class
factory_generated_trajectory = DynamicTrajectory(
        feature_name="Factory-made-mock-trajectory",
        x=mock_data[:, 0],
        y=mock_data[:, 1],
        velocity=mock_data[:, 2],
        acceleration=mock_data[:, 3]
        )

print(factory_generated_trajectory)


          DynamicTrajectory(
             feature_name: Factory-made-mock-trajectory
             trajectory_len: 100
             transposed: False
             dimensions:
                timesteps_indices: (ndarray) shape (100,) range 0 ⟶ 99
                x: (ndarray) shape (100,) range -3.096005928913117 ⟶ 3.011327987100521
                y: (ndarray) shape (100,) range -2.090729284030259 ⟶ 2.8448990219638306
                velocity: (ndarray) shape (100,) range -1.9800630442734126 ⟶ 1.90394937304657
                acceleration: (ndarray) shape (100,) range -2.4288993596829926 ⟶ 2.9451303315287864
          )


## 3. Accessing Trajectory Data

Demonstrate basic data access and manipulation.


In [4]:
# Access individual dimensions
print(f"X position range: [{ trajectory.x.min():.2f}, { trajectory.x.max():.2f}]")
print(f"Y position range: [{ trajectory.y.min():.2f}, { trajectory.y.max():.2f}]")
print(f"Time range: [{ trajectory.timestamps.min():.2f}, { trajectory.timestamps.max():.2f}] seconds")


X position range: [-1.00, 1.00]
Y position range: [-1.00, 1.00]
Time range: [0.00, 4.90] seconds


## 4. Trajectory Slicing

Extract portions of the trajectory.


In [5]:
# Slice trajectory (get timesteps 10-30)
partial_trajectory = trajectory[10:30]
print(f"Original trajectory length: {trajectory.trajectory_len}")
print(f"Partial trajectory length: {partial_trajectory.trajectory_len}")

# Access sliced data
print(f"Partial X range: [{partial_trajectory.x.min():.2f}, {partial_trajectory.x.max():.2f}]")

Original trajectory length: 50
Partial trajectory length: 20
Partial X range: [0.18, 1.00]


## 5. Iterating Through Trajectory Points

Iterate through trajectory data points.


In [6]:
# Iterate through first 5 trajectory points
print("First 5 trajectory points:")
for i, point in enumerate(trajectory):
    if i >= 5:
        break
    print(f"Point {i}: x={point.x:.3f}, y={point.y:.3f}")


First 5 trajectory points:
Point 0: x=0.000, y=1.000
Point 1: x=0.102, y=0.995
Point 2: x=0.203, y=0.979
Point 3: x=0.301, y=0.954
Point 4: x=0.397, y=0.918


## 6. Trajectory Batching

In [7]:
# Create batch trajectories (3 trajectories, 20 timesteps each)
batch_size, time_steps = 3, 20
batch_x = np.random.randn(batch_size, time_steps)
batch_y = np.random.randn(batch_size, time_steps)
batch_frame = np.random.randn(batch_size, time_steps, 10)
batch_timestamps = np.tile(np.arange(time_steps) * 0.1, (batch_size, 1))

@dataclass()
class Simple2DCoordinateTrajectory(BaseTrajectoryDataclass):
    x: np.ndarray
    y: np.ndarray
    frame: np.ndarray
    timestamps: np.ndarray


batch_trajectory = Simple2DCoordinateTrajectory(
    feature_name="batch 2d coordinate",
    x=batch_x,
    y=batch_y,
    frame=batch_frame,
    timestamps=batch_timestamps,
    batch=True
)

print(f"Batch trajectory shape: {batch_trajectory.x.shape}")
print(f"Number of trajectories: {batch_trajectory.x.shape[0]}")
print(f"Timesteps per trajectory: {batch_trajectory.x.shape[1]}")
print(f"Trajectory length: {len(batch_trajectory)}")

print(batch_trajectory)

Batch trajectory shape: (3, 20)
Number of trajectories: 3
Timesteps per trajectory: 20
Trajectory length: 20

          Simple2DCoordinateTrajectory(
             feature_name: batch 2d coordinate
             trajectory_len: 20
             batch: True
             transposed: False
             dimensions:
                timesteps_indices: (ndarray) shape (20,) range 0 ⟶ 19
                x: (ndarray) shape (3, 20) range -2.2032534723940915 ⟶ 1.7706752459389061
                y: (ndarray) shape (3, 20) range -2.626443874593663 ⟶ 2.5798931961697633
                frame: (ndarray) shape (3, 20, 10) range -3.078810689005871 ⟶ 2.6452390873907996
                timestamps: (ndarray) shape (3, 20) range 0.0 ⟶ 1.9000000000000001
          )


## 7. Data Validation

The dataclasses automatically validate data consistency.


In [8]:
# Example of data validation - this will work
trajectory_length = 5
valid_x = np.arange(trajectory_length)
valid_y = np.arange(trajectory_length)
valid_frame = np.random.randn(trajectory_length, 10)
valid_timestamps = np.arange(trajectory_length) * 0.1


# Create trajectory container
valid_trajectory = Simple2DCoordinateTrajectory(
    feature_name="2D coordinate – valid",
    x=valid_x,
    y=valid_y,
    frame=valid_frame,
    timestamps=valid_timestamps
)

print("Valid trajectory created successfully!")
print(f"Trajectory length: {valid_trajectory.trajectory_len}\n")

# Demonstrate error handling with mismatched dimensions
try:
    # This should raise an error due to mismatched array lengths
    invalid_y = np.arange(trajectory_length - 1)  # 4 elements - mismatch!

    invalid_trajectory = Simple2DCoordinateTrajectory(
        feature_name="2D coordinate – invalid",
        x=valid_x,
        y=invalid_y,
        frame=valid_frame,
        timestamps=valid_timestamps
    )

except ValueError as e:
    print(f"Expected error caught: {type(e).__name__}")
    print(e)

Valid trajectory created successfully!
Trajectory length: 5

Expected error caught: ValueError
4 != 5
[TCT error] `2D coordinate – invalid` with container `y` received numpy arrays which do not match the trajectory length


## Summary

This notebook covered the essential usage patterns of TCT dataclasses:

1. **Basic trajectory creation**
2. **Factory-based creation**
3. **Data access** and manipulation methods
4. **Trajectory slicing** for extracting portions of data
5. **Iteration** through trajectory points
6. **Trajectory batching**
7. **Data validation** and error handling

For more examples, refer to the documentation at `documentation/direct_instantiation.md`.
